In [1]:
import glob
import numpy as np
from PIL import Image

images = sorted(glob.glob("Datasets/BSDS500/**/*.jpg", recursive=True))

np.random.seed(1)
data = []
for f in images:
    g = np.array(Image.open(f).convert("L"), dtype=float).reshape(-1)
    data.append(np.random.choice(g, 200, replace=False))
x = np.concatenate(data)
N = len(x)

print("Images:", len(images), " Pixel samples:", N)

k = 3
mu = np.array([45.0, 125.0, 205.0])
var = np.array([1600.0, 1600.0, 1600.0])
pi = np.array([1 / 3, 1 / 3, 1 / 3])


def gaussian(x, mu, var):
    return np.exp(-(x - mu) ** 2 / (2 * var)) / np.sqrt(2 * np.pi * var)


old = -np.inf
for it in range(2000):
    dens = np.zeros((N, k))
    for j in range(k):
        dens[:, j] = pi[j] * gaussian(x, mu[j], var[j])
    total = dens.sum(axis=1)
    ll = np.mean(np.log(total))
    if abs(ll - old) < 1e-7:
        break
    old = ll
    if it % 50 == 0:
        print("Iteration", it, " Log-likelihood =", round(ll, 6))

    resp = dens / total.reshape(-1, 1)

    for j in range(k):
        nk = resp[:, j].sum()
        pi[j] = nk / N
        mu[j] = (resp[:, j] * x).sum() / nk
        var[j] = (resp[:, j] * (x - mu[j]) ** 2).sum() / nk

print("Stopped after", it, "iterations")
print("Final log-likelihood:", round(ll, 6))
for j in range(k):
    print("Component", j + 1, " Mean =", round(mu[j], 2), " Std =", round(np.sqrt(var[j]), 2), " Weight =", round(pi[j], 3))


Images: 500  Pixel samples: 100000
Iteration 0  Log-likelihood = -5.580159
Iteration 50  Log-likelihood = -5.455818
Iteration 100  Log-likelihood = -5.454462
Iteration 150  Log-likelihood = -5.453578
Iteration 200  Log-likelihood = -5.452979
Iteration 250  Log-likelihood = -5.451359
Iteration 300  Log-likelihood = -5.447583
Iteration 350  Log-likelihood = -5.447331
Iteration 400  Log-likelihood = -5.44729
Stopped after 442 iterations
Final log-likelihood: -5.447282
Component 1  Mean = 59.84  Std = 29.59  Weight = 0.361
Component 2  Mean = 132.33  Std = 49.58  Weight = 0.614
Component 3  Mean = 250.58  Std = 4.1  Weight = 0.026
